# 01 - Limpeza e preparação dos dados

Case Olist | Tech Challenge Fase 1

Este é o primeiro notebook do projeto e ele serve de base para todas as análises. A ideia é deixar aqui tudo
que é comum: carregar as tabelas, conferir a qualidade dos dados, arrumar os tipos e montar o modelo estrela.
Depois cada análise parte do resultado que sai daqui.

Uma decisão importante: este notebook não filtra nada. Não corta período e não tira status de pedido.
Isso é de propósito, porque cada análise precisa de um recorte diferente. Quem estuda receita não quer pedido
cancelado, mas quem estuda cancelamento quer justamente ele. Se eu filtrasse aqui, ia atrapalhar as outras
análises.

Os filtros ficam em cada notebook de análise, sempre com a justificativa do lado.

## Como rodar

Baixe os 9 CSVs da Olist no Kaggle e coloque em `data/raw/`. Este notebook grava o resultado em
`data/processed/`, que é de onde os notebooks 02 e 03 vão ler.

In [1]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)

os.makedirs('data/processed', exist_ok=True)

## 1. Carregando as tabelas

In [2]:
customers = pd.read_csv('data/raw/olist_customers_dataset.csv')
order_items = pd.read_csv('data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('data/raw/olist_orders_dataset.csv')
products = pd.read_csv('data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('data/raw/olist_sellers_dataset.csv')
cat_translation = pd.read_csv('data/raw/product_category_name_translation.csv')

In [3]:
tabelas = {
    'customers': customers,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'cat_translation': cat_translation,
}

for nome, tabela in tabelas.items():
    print(nome, tabela.shape)

customers (99441, 5)
order_items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
orders (99441, 8)
products (32951, 9)
sellers (3095, 4)
cat_translation (71, 2)


Não carreguei a tabela `geolocation`. Ela tem latitude e longitude por prefixo de CEP e traz muita linha
repetida, e as análises usam a UF do cliente, que já vem em `customers`. Se alguma análise precisar de mapa
depois, ela pode carregar essa tabela por conta.

## 2. Conferindo a qualidade dos dados

Antes de montar qualquer coisa, fui olhar o que tem de errado na base.

In [4]:
for nome, tabela in tabelas.items():
    print(nome, '- nulos:', tabela.isnull().sum().sum())

customers - nulos: 0
order_items - nulos: 0
payments - nulos: 0
reviews - nulos: 145903
orders - nulos: 4908
products - nulos: 2448
sellers - nulos: 0
cat_translation - nulos: 0


Três tabelas têm nulo. Fui ver em quais colunas.

In [5]:
for nome in ['orders', 'products', 'reviews']:
    print('---', nome, '---')
    nulos = tabelas[nome].isnull().sum()
    print(nulos[nulos > 0])
    print()

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64



Olhando coluna por coluna, decidi não preencher quase nada:

- Em `orders` os nulos são datas de entrega de pedidos que não foram entregues. Faz sentido estarem vazias, e
  quem for analisar entrega vai trabalhar só com os entregues mesmo.
- Em `products` são medidas e descrição. A única que atrapalha é a categoria, que eu preencho como `unknown`.
- Em `reviews` são os campos de comentário, que o cliente não é obrigado a escrever.

O importante é que nenhuma coluna usada em cálculo de valor está vazia.

In [6]:
print('nulos em price:', order_items['price'].isnull().sum())
print('nulos em freight_value:', order_items['freight_value'].isnull().sum())
print('nulos em order_purchase_timestamp:', orders['order_purchase_timestamp'].isnull().sum())

nulos em price: 0
nulos em freight_value: 0
nulos em order_purchase_timestamp: 0


In [7]:
for nome, tabela in tabelas.items():
    print(nome, '- linhas duplicadas:', tabela.duplicated().sum())

customers - linhas duplicadas: 0
order_items - linhas duplicadas: 0
payments - linhas duplicadas: 0


reviews - linhas duplicadas: 0


orders - linhas duplicadas: 0
products - linhas duplicadas: 0
sellers - linhas duplicadas: 0
cat_translation - linhas duplicadas: 0


Nenhuma tabela tem linha inteira duplicada.

### Arrumando as datas

In [8]:
colunas_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

for col in colunas_data:
    orders[col] = pd.to_datetime(orders[col])

orders[colunas_data].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Aqui tem um detalhe que vale registrar, porque ele muda resultado depois.

A data prometida de entrega (`order_estimated_delivery_date`) vem sem hora, sempre meia-noite. Já a data
real da entrega vem com hora cheia. Se alguém comparar as duas direto, um pedido entregue no dia certo às 15h
aparece como atrasado, porque 15h é maior que meia-noite.

In [9]:
sem_hora = (orders['order_estimated_delivery_date'].dt.time.astype(str) == '00:00:00').all()
print('data prometida sempre à meia-noite?', sem_hora)

entregues = orders[orders['order_delivered_customer_date'].notna()]
falso_atraso = (
    (entregues['order_delivered_customer_date'] > entregues['order_estimated_delivery_date']) &
    (entregues['order_delivered_customer_date'].dt.normalize() <= entregues['order_estimated_delivery_date'].dt.normalize())
).sum()

print('pedidos entregues no dia prometido, mas depois da meia-noite:', falso_atraso)

data prometida sempre à meia-noite? True
pedidos entregues no dia prometido, mas depois da meia-noite: 1292


São 1.292 pedidos. Se a comparação for feita errado, o percentual de atraso da base sobe de 6,8% para
8,1%, o que é bastante.

Combinado do projeto: atraso é quando a data da entrega é posterior à data prometida, comparando só o dia e
ignorando a hora. Deixo essa coluna já pronta aqui para ninguém precisar refazer a conta.

In [10]:
# Trago o estado e o identificador real do cliente para a tabela de pedidos.
# O customer_id muda a cada pedido, quem identifica a pessoa é o customer_unique_id.
orders = orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id', how='left'
)

print('pedidos sem cliente correspondente:', orders['customer_unique_id'].isnull().sum())

pedidos sem cliente correspondente: 0


In [11]:
orders['atrasou'] = (
    orders['order_delivered_customer_date'].dt.normalize() >
    orders['order_estimated_delivery_date'].dt.normalize()
)

orders['dias_entrega'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

orders['mes'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

In [12]:
# Conferindo se tem data de entrega anterior à data da compra
invertidos = orders['dias_entrega'] < 0
print('pedidos com data invertida:', invertidos.sum())

pedidos com data invertida: 0


Zero. As datas estão coerentes, não precisei tratar nada aqui.

### Panorama da base, sem filtrar nada

Registro aqui os números da base inteira, para servir de referência a todo mundo. Quem for analisar decide
depois o que cortar.

In [13]:
print('primeira compra:', orders['order_purchase_timestamp'].min())
print('última compra:', orders['order_purchase_timestamp'].max())
print('total de pedidos:', len(orders))
print()
print(orders['order_status'].value_counts())

primeira compra: 2016-09-04 21:15:19
última compra: 2018-10-17 17:30:18
total de pedidos: 99441

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [14]:
pedidos_por_mes = orders['mes'].value_counts().sort_index()
print(pedidos_por_mes)

mes
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Name: count, dtype: int64


Dois pontos que aparecem aqui e que cada análise vai ter que decidir o que fazer:

As pontas da série são fracas. Setembro de 2016 tem 4 pedidos, outubro tem 324, novembro não existe na base e
dezembro tem 1. No outro extremo, setembro e outubro de 2018 somam 20 pedidos.

E tem pedido que não virou venda. Vale conferir quantos deles têm produto associado antes de somar receita.

In [15]:
for status in ['canceled', 'unavailable', 'shipped', 'processing', 'invoiced', 'delivered']:
    do_status = orders[orders['order_status'] == status]
    com_item = do_status['order_id'].isin(order_items['order_id']).mean() * 100
    print(status.ljust(12), '- pedidos:', len(do_status), '- com produto:', round(com_item, 1), '%')

canceled     - pedidos: 625 - com produto: 73.8 %
unavailable  - pedidos: 609 - com produto: 1.0 %
shipped      - pedidos: 1107 - com produto: 99.9 %
processing   - pedidos: 301 - com produto: 100.0 %
invoiced     - pedidos: 314 - com produto: 99.4 %
delivered    - pedidos: 96478 - com produto: 100.0 %


Isso confirma o problema: nenhum pedido `unavailable` tem produto associado e só 77% dos `canceled` têm.
Os outros status têm 100%.

Não vou excluir nada aqui, só deixar o dado registrado. Quem for calcular receita vai querer tirar
`canceled` e `unavailable`, e quem for estudar cancelamento vai querer justamente esses. Cada notebook decide.

## 3. Preparando as tabelas

Aqui tem o cuidado que mais afeta os números do projeto inteiro.

In [16]:
# Categoria traduzida, com as que faltam marcadas como unknown
dim_produtos = products.merge(cat_translation, on='product_category_name', how='left')
dim_produtos['product_category_name_english'] = dim_produtos['product_category_name_english'].fillna('unknown')

print('categorias diferentes:', dim_produtos['product_category_name_english'].nunique())

categorias diferentes: 72


Um mesmo pedido pode ter várias linhas em `payments`, quando o cliente divide o pagamento, e também em
`reviews`. Se eu juntar essas tabelas sem agregar antes, cada item do pedido vira várias linhas e a receita
conta em dobro.

In [17]:
print('payments - linhas:', len(payments), '| pedidos:', payments['order_id'].nunique())
print('reviews  - linhas:', len(reviews), '| pedidos:', reviews['order_id'].nunique())

payments - linhas: 103886 | pedidos: 99440
reviews  - linhas: 99224 | pedidos: 98673


In [18]:
# Agregando por pedido antes de juntar
pagamento_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()
pagamento_por_pedido.columns = ['order_id', 'total_pago']

tipo_pagamento = payments.sort_values('payment_value', ascending=False).groupby('order_id').first().reset_index()
tipo_pagamento = tipo_pagamento[['order_id', 'payment_type', 'payment_installments']]

nota_por_pedido = reviews.groupby('order_id')['review_score'].mean().reset_index()

## 4. Montando o modelo estrela

Organizei os dados como vimos na aula 3 de Fundamentos: uma tabela fato com o que acontece, e tabelas
dimensão com as informações de apoio. A tabela fato tem uma linha por item vendido.

In [19]:
fato_itens = order_items.merge(orders, on='order_id', how='left')
fato_itens = fato_itens.merge(pagamento_por_pedido, on='order_id', how='left')
fato_itens = fato_itens.merge(tipo_pagamento, on='order_id', how='left')
fato_itens = fato_itens.merge(nota_por_pedido, on='order_id', how='left')
fato_itens = fato_itens.merge(dim_produtos[['product_id', 'product_category_name_english']], on='product_id', how='left')
fato_itens = fato_itens.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')

# customer_state e customer_unique_id já vieram junto com orders, não precisa juntar de novo
print('tabela fato:', fato_itens.shape)

tabela fato: (112650, 25)


In [20]:
# A receita de cada item é o preço do produto mais o frete
fato_itens['receita'] = fato_itens['price'] + fato_itens['freight_value']

### Validando antes de salvar

In [21]:
print('linhas em order_items:', len(order_items))
print('linhas na tabela fato:', len(fato_itens))
print('continua igual?', len(fato_itens) == len(order_items))
print()
print('soma de price na origem:', round(order_items['price'].sum(), 2))
print('soma de price na fato  :', round(fato_itens['price'].sum(), 2))
print()
print('itens sem categoria:', fato_itens['product_category_name_english'].isnull().sum())
print('itens sem estado do cliente:', fato_itens['customer_state'].isnull().sum())

linhas em order_items: 112650
linhas na tabela fato: 112650
continua igual? True

soma de price na origem: 13591643.7
soma de price na fato  : 13591643.7

itens sem categoria: 0
itens sem estado do cliente: 0


As três checagens passam: o número de linhas não mudou depois dos merges, a soma de preço bate com a
origem e nenhuma chave ficou sem correspondência. Se alguma dessas falhar, tem duplicação em algum merge.

## 5. Salvando o resultado

In [22]:
dim_clientes = customers.copy()
dim_vendedores = sellers.copy()

dim_data = orders[['order_id', 'order_purchase_timestamp', 'mes']].copy()
dim_data['ano'] = dim_data['order_purchase_timestamp'].dt.year
dim_data['dia_semana'] = dim_data['order_purchase_timestamp'].dt.dayofweek

In [23]:
fato_itens.to_csv('data/processed/fato_itens.csv', index=False)
orders.to_csv('data/processed/pedidos_tratados.csv', index=False)
dim_produtos.to_csv('data/processed/dim_produtos.csv', index=False)
dim_clientes.to_csv('data/processed/dim_clientes.csv', index=False)
dim_vendedores.to_csv('data/processed/dim_vendedores.csv', index=False)
dim_data.to_csv('data/processed/dim_data.csv', index=False)

print('arquivos salvos em data/processed/')

arquivos salvos em data/processed/


## O que foi decidido aqui

Resumo das escolhas deste notebook, para quem for usar a base:

- Receita = preço do produto + frete. As duas partes somadas são o valor transacionado no item.
- Atraso é comparado só pela data, ignorando a hora, porque a data prometida vem sem hora. A coluna
  `atrasou` já vem pronta.
- `payments` e `reviews` foram agregados por pedido antes dos merges, para não duplicar receita.
- Categoria sem tradução virou `unknown`.
- Nenhum pedido foi excluído e nenhum período foi cortado. Isso é responsabilidade de cada análise.
- `geolocation` não entra, por ter muita repetição e não ser necessária para a UF do cliente.

Os arquivos ficam em `data/processed/`. O notebook 02 analisa crescimento e receita, e o 03 analisa a
concentração e a entrega.